# Muhtemel Ask - 02 FINALIZE
Run all cells. For normal use, change only `EPISODE`. The notebook validates the locked PREPARE schema, translation-pack manifest, translated ZIP, semantic alignment and every hard QA gate before it creates any staged output. Final files are replaced atomically one by one; the finalization report is written last as the commit marker.

In [ ]:
EPISODE = 12  # @param {type:"integer"}

if isinstance(EPISODE, bool) or not isinstance(EPISODE, int) or EPISODE < 1:
    raise ValueError("EPISODE must be a positive integer")

In [ ]:
# Advanced settings - normal use does not require changes.
CREATE_MKV = False  # @param {type:"boolean"}
STRICT_QA = True  # @param {type:"boolean"}
FORCE = False  # @param {type:"boolean"}

for _name, _value in {"CREATE_MKV": CREATE_MKV, "STRICT_QA": STRICT_QA, "FORCE": FORCE}.items():
    if not isinstance(_value, bool):
        raise ValueError(f"{_name} must be True or False")

## Mount Drive and prepare the Colab runtime

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import shutil
import subprocess
import sys

SYSTEM_ROOT = Path("/content/drive/MyDrive/Muhtemel_Ask_Subtitles/SYSTEM")
if not (SYSTEM_ROOT / "src").is_dir():
    raise FileNotFoundError(
        f"System files were not found at {SYSTEM_ROOT}. Follow 00_README_FIRST.md."
    )
requirements_path = SYSTEM_ROOT / "requirements-colab.txt"
if not requirements_path.is_file():
    raise FileNotFoundError(f"Missing Colab requirements: {requirements_path}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
    check=True,
)
if CREATE_MKV and (shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
if str(SYSTEM_ROOT) not in sys.path:
    sys.path.insert(0, str(SYSTEM_ROOT))
print("Runtime ready.")

## Load the production modules and safety helpers

In [ ]:
from datetime import datetime, timezone
import hashlib
import json
import os
import tempfile
import uuid

import yaml
from openpyxl import load_workbook

from src.batches import validate_translation_pack
from src.download import load_valid_stage_marker
from src.mux import mux_softsubs
from src.review import REVIEW_COLUMNS, create_review_xlsx
from src.schema import atomic_write_json, load_schema, read_json, sha256_file
from src.srt import (
    assert_srt_roundtrip,
    build_entries,
    parse_srt,
    visible_length,
    write_srt,
)
from src.subtitle_qa import assert_final_qa, run_subtitle_qa
from src.translation_validation import (
    TranslationValidationError,
    load_and_validate_translated_zip,
)

VALIDATION_ZERO_FIELDS = (
    "missing_translation_count",
    "duplicate_translation_count",
    "extra_translation_count",
    "positional_translation_mismatch_count",
    "schema_mismatch_count",
    "uid_mismatch_count",
    "order_mismatch_count",
    "block_count_mismatch_count",
    "timing_mismatch_count",
    "episode_mismatch_count",
    "schema_version_mismatch_count",
    "block_index_mismatch_count",
    "record_shape_mismatch_count",
    "empty_text_count",
    "special_name_mismatch_count",
    "numeric_mismatch_count",
    "religious_expression_mismatch_count",
    "allah_preservation_mismatch_count",
    "translation_report_mismatch_count",
    "batch_mismatch_count",
)

QA_ZERO_FIELDS = (
    "missing_translation_count",
    "duplicate_translation_count",
    "extra_translation_count",
    "positional_translation_mismatch_count",
    "overlap_count",
    "early_start_count",
    "early_end_count",
    "empty_text_count",
    "more_than_two_lines_count",
    "line_over_84_count",
    "mixed_speaker_block_count",
    "unresolved_internal_gap_count",
    "schema_mismatch_count",
    "uid_mismatch_count",
    "order_mismatch_count",
    "timing_mismatch_count",
    "timing_error_count",
    "special_name_mismatch_count",
    "numeric_mismatch_count",
    "money_mismatch_count",
    "religious_expression_mismatch_count",
    "allah_preservation_failure_count",
    "anchor_mismatch_count",
    "utf8_error_count",
    "srt_structure_error_count",
    "music_speech_handling_count",
)

def require_inside(path, root, label):
    resolved = Path(path).resolve()
    resolved_root = Path(root).resolve()
    if resolved != resolved_root and resolved_root not in resolved.parents:
        raise RuntimeError(f"{label} escapes the episode workspace: {resolved}")
    return resolved

def require_file(path, label):
    value = Path(path)
    if not value.is_file():
        raise FileNotFoundError(f"{label} not found: {value}")
    if value.stat().st_size <= 0:
        raise RuntimeError(f"{label} is empty: {value}")
    return value

def file_record(path, episode_root):
    value = require_file(path, "Published artifact")
    return {
        "relative_path": value.resolve().relative_to(episode_root.resolve()).as_posix(),
        "size_bytes": value.stat().st_size,
        "sha256": sha256_file(value),
    }

def notebook_source_sha256(path):
    notebook = read_json(path)
    cells = [
        {"cell_type": cell.get("cell_type"), "source": "".join(cell.get("source", []))}
        for cell in notebook.get("cells", [])
    ]
    payload = json.dumps(
        cells, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def assert_zero_fields(report, fields, label):
    failures = []
    for field in fields:
        if field not in report:
            failures.append(f"{field}=<missing>")
            continue
        value = report[field]
        if isinstance(value, bool) or not isinstance(value, int) or value != 0:
            failures.append(f"{field}={value!r}")
    if failures:
        raise RuntimeError(label + " failed; required zero: " + ", ".join(failures))

def assert_entry_contract(entries, blocks, language, hard_limit):
    if len(entries) != len(blocks):
        raise RuntimeError(
            f"{language} SRT count mismatch: expected {len(blocks)}, got {len(entries)}"
        )
    for position, (entry, block) in enumerate(zip(entries, blocks), start=1):
        expected = (block["block_index"], block["start_ms"], block["end_ms"])
        actual = (entry.index, entry.start_ms, entry.end_ms)
        if actual != expected:
            raise RuntimeError(
                f"{language} timing/index mismatch at {block['block_uid']}: "
                f"expected {expected}, got {actual}"
            )
        lines = entry.text.replace("\r\n", "\n").replace("\r", "\n").split("\n")
        if len(lines) > 2:
            raise RuntimeError(f"{language} has more than 2 lines at {block['block_uid']}")
        if any(visible_length(line) > hard_limit for line in lines):
            raise RuntimeError(f"{language} has a line over {hard_limit} characters at {block['block_uid']}")

def assert_identical_timings(tr_entries, id_entries):
    tr_timing = [(entry.index, entry.start_ms, entry.end_ms) for entry in tr_entries]
    id_timing = [(entry.index, entry.start_ms, entry.end_ms) for entry in id_entries]
    if tr_timing != id_timing:
        for position, (tr_value, id_value) in enumerate(zip(tr_timing, id_timing), start=1):
            if tr_value != id_value:
                raise RuntimeError(
                    f"TR/ID timing mismatch at block {position}: TR={tr_value}, ID={id_value}"
                )
        raise RuntimeError("TR/ID timing list lengths differ")

def verify_review_xlsx(path):
    workbook = load_workbook(path, read_only=True, data_only=False)
    try:
        if workbook.sheetnames != ["Kontrol"]:
            raise RuntimeError(f"Review workbook sheet mismatch: {workbook.sheetnames!r}")
        sheet = workbook["Kontrol"]
        headers = tuple(sheet.cell(1, column).value for column in range(1, 11))
        if headers != REVIEW_COLUMNS:
            raise RuntimeError(f"Review workbook header mismatch: {headers!r}")
        return max(0, sheet.max_row - 1)
    finally:
        workbook.close()

def verified_source_from_marker(source_dir):
    marker_path = require_file(source_dir / "download.done.json", "Download marker")
    raw_marker = read_json(marker_path)
    if not isinstance(raw_marker, dict) or not isinstance(raw_marker.get("input_sha256"), str):
        raise RuntimeError(f"Malformed download marker: {marker_path}")
    marker = load_valid_stage_marker(
        marker_path,
        stage="download",
        input_sha256=raw_marker["input_sha256"],
        required_output_keys=("video", "metadata"),
        allowed_root=source_dir,
    )
    if marker is None:
        raise RuntimeError("Source video/download marker hash validation failed")
    source_video = require_inside(
        marker["outputs"]["video"]["path"], source_dir, "Source video"
    )
    require_file(source_video, "Source video")
    return source_video, marker

def atomic_publish(staged, target):
    staged = require_file(staged, "Staged artifact")
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if staged.parent.resolve() != target.parent.resolve():
        raise RuntimeError("Staged and final artifact must share a directory for atomic replace")
    expected_hash = sha256_file(staged)
    os.replace(staged, target)
    if sha256_file(target) != expected_hash:
        raise RuntimeError(f"Published artifact hash changed: {target}")
    return target

def show_translation_failure(error):
    print(f"TRANSLATION VALIDATION FAILED: {error}")
    report = getattr(error, "report", None)
    if report is None:
        return
    for issue in report.errors[:20]:
        print(json.dumps(issue.to_dict(), ensure_ascii=False, default=str))
    if len(report.errors) > 20:
        print(f"... {len(report.errors) - 20} more validation errors")

## Resolve this episode's exact inputs and validate translations
No wildcard or block-index-only matching is used. The PREPARE pack supplies the authoritative batch topology; `block_uid` and the immutable schema hash supply identity.

In [ ]:
with (SYSTEM_ROOT / "config/series.yaml").open(encoding="utf-8") as handle:
    series_config = yaml.safe_load(handle)
with (SYSTEM_ROOT / "config/names.yaml").open(encoding="utf-8") as handle:
    names_config = yaml.safe_load(handle)
with (SYSTEM_ROOT / "config/religious_terms.yaml").open(encoding="utf-8") as handle:
    religious_config = yaml.safe_load(handle)
if not all(isinstance(value, dict) for value in (series_config, names_config, religious_config)):
    raise RuntimeError("One or more configuration files are not YAML mappings")

my_drive = Path("/content/drive/MyDrive").resolve()
drive_root = require_inside(Path(series_config["drive_root"]), my_drive, "Configured Drive root")
episode_template = series_config["episode_folder_template"]
episode_name = episode_template.format(episode=EPISODE)
EPISODE_ROOT = require_inside(
    drive_root / "EPISODES" / episode_name, drive_root / "EPISODES", "Episode root"
)
if not EPISODE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Episode workspace not found: {EPISODE_ROOT}. Run 01_PREPARE first."
    )
DIRS = {name: EPISODE_ROOT / name for name in (
    "source", "prepare", "translation_input",
    "translation_output", "review", "final",
)}
for required_dir in ("source", "prepare", "translation_input", "translation_output"):
    if not DIRS[required_dir].is_dir():
        raise FileNotFoundError(f"Required episode directory is missing: {DIRS[required_dir]}")
for output_dir in (DIRS["review"], DIRS["final"]):
    output_dir.mkdir(parents=True, exist_ok=True)

schema_path = require_file(DIRS["prepare"] / "schema.json", "Locked episode schema")
translation_pack_path = require_file(
    DIRS["translation_input"]
    / series_config["translation_pack_template"].format(episode=EPISODE),
    "PREPARE translation pack",
)
translated_zip_path = require_file(
    DIRS["translation_output"]
    / series_config["translated_pack_template"].format(episode=EPISODE),
    "Work Ultra translated ZIP",
)
for input_path, label in (
    (schema_path, "Schema"),
    (translation_pack_path, "Translation pack"),
    (translated_zip_path, "Translated ZIP"),
):
    require_inside(input_path, EPISODE_ROOT, label)

input_hash_paths = {
    "schema_file_sha256": schema_path,
    "translation_pack_sha256": translation_pack_path,
    "translated_zip_sha256": translated_zip_path,
    "series_config_sha256": SYSTEM_ROOT / "config/series.yaml",
    "names_config_sha256": SYSTEM_ROOT / "config/names.yaml",
    "religious_config_sha256": SYSTEM_ROOT / "config/religious_terms.yaml",
    "batches_module_sha256": SYSTEM_ROOT / "src/batches.py",
    "download_module_sha256": SYSTEM_ROOT / "src/download.py",
    "schema_module_sha256": SYSTEM_ROOT / "src/schema.py",
    "translation_validation_module_sha256": SYSTEM_ROOT / "src/translation_validation.py",
    "subtitle_qa_module_sha256": SYSTEM_ROOT / "src/subtitle_qa.py",
    "srt_module_sha256": SYSTEM_ROOT / "src/srt.py",
    "review_module_sha256": SYSTEM_ROOT / "src/review.py",
    "mux_module_sha256": SYSTEM_ROOT / "src/mux.py",
}
for hash_path in input_hash_paths.values():
    require_file(hash_path, "Finalization input")
input_hashes = {key: sha256_file(path) for key, path in input_hash_paths.items()}
input_hashes["finalize_notebook_source_sha256"] = notebook_source_sha256(
    SYSTEM_ROOT / "02_FINALIZE.ipynb"
)
schema = load_schema(schema_path, expected_episode=EPISODE)
pack_manifest = validate_translation_pack(translation_pack_path, expected_schema=schema)
try:
    validation = load_and_validate_translated_zip(
        schema,
        translated_zip_path,
        input_manifest=pack_manifest,
        known_names=tuple(names_config.get("canonical_names", ())),
        forbidden_name_variants=names_config.get("forbidden_variants", {}),
        source_name_variants=names_config.get("source_variants", {}),
        semantic_window=6,
        raise_on_error=True,
    )
except TranslationValidationError as error:
    show_translation_failure(error)
    raise

for key, input_path in input_hash_paths.items():
    if sha256_file(input_path) != input_hashes[key]:
        raise RuntimeError(f"Input changed while it was being validated: {input_path}")
if notebook_source_sha256(SYSTEM_ROOT / "02_FINALIZE.ipynb") != input_hashes["finalize_notebook_source_sha256"]:
    raise RuntimeError("FINALIZE notebook source changed while inputs were validated")

validation_report = validation.to_dict()
if validation.ok is not True or validation_report.get("ok") is not True:
    raise RuntimeError("Translation validator did not return an explicit PASS")
assert_zero_fields(validation_report, VALIDATION_ZERO_FIELDS, "Translation validation")
if validation_report["expected_block_count"] != schema["block_count"]:
    raise RuntimeError("Validator expected block count differs from the locked schema")
if validation_report["output_block_count"] != schema["block_count"]:
    raise RuntimeError("Translated output block count differs from the locked schema")
if validation_report["tr_block_count"] != schema["block_count"]:
    raise RuntimeError("Turkish translation count differs from the locked schema")
if validation_report["id_block_count"] != schema["block_count"]:
    raise RuntimeError("Indonesian translation count differs from the locked schema")
ordered_records = validation.ordered_records(schema)
expected_uids = [block["block_uid"] for block in schema["blocks"]]
if [record["block_uid"] for record in ordered_records] != expected_uids:
    raise RuntimeError("Validated records are not in immutable schema UID order")

print(f"Episode: {episode_name}")
print(f"Locked blocks: {schema['block_count']:,}")
print(f"Schema SHA-256: {schema['schema_sha256']}")
print(f"Translated ZIP SHA-256: {input_hashes['translated_zip_sha256']}")
print("Translation ZIP, UID/hash/order and semantic alignment: PASS")

## Run complete deterministic subtitle QA
All required zero-count gates remain mandatory even when `STRICT_QA` is disabled. `STRICT_QA` additionally requires the module's complete hard-gate verdict, including any future hard checks. Reading-speed and manual-review counts remain visible warnings, as intended.

In [ ]:
subtitle_config = series_config["subtitle"]
qa_report = run_subtitle_qa(
    schema["blocks"],
    validation,
    names_config=names_config,
    religious_config=religious_config,
    preferred_max_cps=float(subtitle_config["preferred_max_cps"]),
    line_limit=int(subtitle_config["qa_max_chars_per_line"]),
)
assert_zero_fields(qa_report, QA_ZERO_FIELDS, "Final subtitle QA")
if qa_report.get("tr_block_count") != schema["block_count"]:
    raise RuntimeError("QA Turkish block count differs from the locked schema")
if qa_report.get("id_block_count") != schema["block_count"]:
    raise RuntimeError("QA Indonesian block count differs from the locked schema")
if qa_report.get("timings_identical") is not True:
    raise RuntimeError("QA did not confirm identical Turkish/Indonesian timings")
if STRICT_QA:
    assert_final_qa(qa_report)
elif qa_report.get("passed") is not True:
    # Disabling strict mode never bypasses the explicit mandatory gates above.
    raise RuntimeError("QA report did not return PASS after mandatory checks")

print("Final hard QA: PASS")
print(f"Review-required blocks: {qa_report['review_required_count']:,}")
print(f"High-CPS warnings: {qa_report['high_cps_count']:,}")

## Build, round-trip verify and publish
SRT timings come only from `schema.json`. Turkish and Indonesian files are independently parsed after writing and compared to the locked schema. Infuse-ready `-id.srt` and `-tr.srt` sidecars are also written beside the verified source video, without copying the video. The review workbook targets at most 5% total uncertainty rows while never dropping an explicit translator review request. If enabled, MKV subtitle tracks are extracted and compared exactly, and compressed video/audio stream hashes must match the source.

In [ ]:
tr_final_path = DIRS["final"] / f"{episode_name}.tr-final.srt"
id_final_path = DIRS["final"] / f"{episode_name}.id-final.srt"
review_final_path = DIRS["review"] / f"{episode_name}_review.xlsx"
mkv_final_path = DIRS["final"] / f"{episode_name} - Endonezce + Turkce.mkv"
finalization_report_path = DIRS["final"] / f"{episode_name}_FINALIZATION_REPORT.json"
source_video, source_marker = verified_source_from_marker(DIRS["source"])
source_marker_path = DIRS["source"] / "download.done.json"
tr_sidecar_path = source_video.with_name(f"{source_video.stem}-tr.srt")
id_sidecar_path = source_video.with_name(f"{source_video.stem}-id.srt")
require_inside(tr_sidecar_path, DIRS["source"], "Turkish Infuse sidecar")
require_inside(id_sidecar_path, DIRS["source"], "Indonesian Infuse sidecar")
input_hash_paths["source_video_sha256"] = source_video
input_hash_paths["download_marker_sha256"] = source_marker_path
input_hashes["source_video_sha256"] = sha256_file(source_video)
input_hashes["download_marker_sha256"] = sha256_file(source_marker_path)
output_paths = {
    "tr_srt": tr_final_path,
    "id_srt": id_final_path,
    "tr_infuse_sidecar": tr_sidecar_path,
    "id_infuse_sidecar": id_sidecar_path,
    "review_xlsx": review_final_path,
}
if CREATE_MKV:
    output_paths["mkv"] = mkv_final_path

REPORT_INPUT_SPECS = {
    "schema": ("schema_file_sha256", schema_path),
    "translation_pack": ("translation_pack_sha256", translation_pack_path),
    "translated_zip": ("translated_zip_sha256", translated_zip_path),
    "download_marker": ("download_marker_sha256", source_marker_path),
    "source_video": ("source_video_sha256", source_video),
}

def assert_current_input_hashes(context):
    mismatches = []
    for key, input_path in input_hash_paths.items():
        try:
            actual = sha256_file(require_file(input_path, "Finalization input"))
        except Exception as error:
            mismatches.append(f"{key}: unreadable ({error})")
            continue
        if actual != input_hashes[key]:
            mismatches.append(
                f"{key}: expected {input_hashes[key]}, got {actual}"
            )
    notebook_actual = notebook_source_sha256(SYSTEM_ROOT / "02_FINALIZE.ipynb")
    notebook_expected = input_hashes["finalize_notebook_source_sha256"]
    if notebook_actual != notebook_expected:
        mismatches.append(
            f"finalize_notebook_source_sha256: expected {notebook_expected}, "
            f"got {notebook_actual}"
        )
    if mismatches:
        raise RuntimeError(
            f"Finalization inputs changed {context}; refusing PASS:\n- "
            + "\n- ".join(mismatches)
        )

def report_input_file_records():
    records = {}
    for label, (hash_key, input_path) in REPORT_INPUT_SPECS.items():
        record = file_record(input_path, EPISODE_ROOT)
        if record["sha256"] != input_hashes[hash_key]:
            raise RuntimeError(
                f"Report input hash mismatch for {label}: "
                f"expected {input_hashes[hash_key]}, got {record['sha256']}"
            )
        records[label] = record
    return records

def current_report_or_reason():
    if not finalization_report_path.is_file():
        return None, "no prior finalization report"
    try:
        prior = read_json(finalization_report_path)
        if prior.get("report_version") != 1:
            return None, "prior report version mismatch"
        expected_identity = {
            "episode": EPISODE,
            "schema_version": schema["schema_version"],
            "schema_sha256": schema["schema_sha256"],
            "block_count": schema["block_count"],
        }
        for key, value in expected_identity.items():
            if prior.get(key) != value:
                return None, f"prior report {key} mismatch"
        if prior.get("status") != "PASS":
            return None, "prior report status is not PASS"
        if prior.get("input_hashes") != input_hashes:
            return None, "input hashes changed"
        if prior.get("input_files") != report_input_file_records():
            return None, "prior input-file records differ from current hashed inputs"
        if prior.get("settings") != {
            "create_mkv": CREATE_MKV, "strict_qa": STRICT_QA
        }:
            return None, "finalization settings changed"
        assert_zero_fields(prior["translation_validation"], VALIDATION_ZERO_FIELDS, "Prior translation validation")
        assert_zero_fields(prior["subtitle_qa"], QA_ZERO_FIELDS, "Prior subtitle QA")
        records = prior.get("outputs")
        if not isinstance(records, dict) or set(records) != set(output_paths):
            return None, "prior output set differs"
        for key, path in output_paths.items():
            record = records.get(key)
            if not isinstance(record, dict):
                return None, f"missing prior output record: {key}"
            expected_relative = path.resolve().relative_to(EPISODE_ROOT.resolve()).as_posix()
            if record.get("relative_path") != expected_relative:
                return None, f"prior output path mismatch: {key}"
            if not path.is_file() or path.stat().st_size != record.get("size_bytes"):
                return None, f"prior output missing/size mismatch: {key}"
            if sha256_file(path) != record.get("sha256"):
                return None, f"prior output hash mismatch: {key}"
        tr_existing = parse_srt(tr_final_path)
        id_existing = parse_srt(id_final_path)
        tr_sidecar_existing = parse_srt(tr_sidecar_path)
        id_sidecar_existing = parse_srt(id_sidecar_path)
        assert_entry_contract(tr_existing, schema["blocks"], "TR", int(subtitle_config["qa_max_chars_per_line"]))
        assert_entry_contract(id_existing, schema["blocks"], "ID", int(subtitle_config["qa_max_chars_per_line"]))
        assert_entry_contract(tr_sidecar_existing, schema["blocks"], "TR sidecar", int(subtitle_config["qa_max_chars_per_line"]))
        assert_entry_contract(id_sidecar_existing, schema["blocks"], "ID sidecar", int(subtitle_config["qa_max_chars_per_line"]))
        assert_identical_timings(tr_existing, id_existing)
        if tr_sidecar_existing != tr_existing or id_sidecar_existing != id_existing:
            return None, "Infuse sidecars differ from canonical SRT files"
        verify_review_xlsx(review_final_path)
        if CREATE_MKV:
            mux_check = prior.get("mkv_verification", {})
            if not (
                mux_check.get("verified") is True
                and mux_check.get("video_audio_stream_copy") is True
                and mux_check.get("roundtrip", {}).get("exact") is True
                and mux_check.get("stream_hashes", {}).get("checked") is True
                and mux_check.get("stream_hashes", {}).get("match") is True
            ):
                return None, "prior MKV verification is incomplete"
        elif mkv_final_path.exists():
            return None, "unexpected canonical MKV exists while CREATE_MKV=False"
        assert_current_input_hashes("while accepting a prior finalization")
        return prior, "verified prior finalization"
    except Exception as error:
        return None, f"prior finalization verification failed: {error}"

prior_report, resume_reason = (None, "FORCE=True") if FORCE else current_report_or_reason()
if prior_report is not None:
    print("Verified finalization already exists; no files were replaced.")
    finalization_report = prior_report
else:
    print(f"Building final outputs ({resume_reason}).")
    assert_current_input_hashes("before staging final outputs")

    target_chars = int(subtitle_config["target_chars_per_line"])
    hard_limit = int(subtitle_config["qa_max_chars_per_line"])
    tr_entries = build_entries(
        schema["blocks"], validation.validated_by_uid, language="tr",
        target=target_chars, max_lines=2, hard_limit=hard_limit,
    )
    id_entries = build_entries(
        schema["blocks"], validation.validated_by_uid, language="id",
        target=target_chars, max_lines=2, hard_limit=hard_limit,
    )
    assert_entry_contract(tr_entries, schema["blocks"], "TR", hard_limit)
    assert_entry_contract(id_entries, schema["blocks"], "ID", hard_limit)
    assert_identical_timings(tr_entries, id_entries)

    run_id = uuid.uuid4().hex
    tr_stage = DIRS["final"] / f".{tr_final_path.stem}.{run_id}.staged.srt"
    id_stage = DIRS["final"] / f".{id_final_path.stem}.{run_id}.staged.srt"
    tr_sidecar_stage = DIRS["source"] / f".{tr_sidecar_path.stem}.{run_id}.staged.srt"
    id_sidecar_stage = DIRS["source"] / f".{id_sidecar_path.stem}.{run_id}.staged.srt"
    review_stage = DIRS["review"] / f".{review_final_path.stem}.{run_id}.staged.xlsx"
    mkv_stage = DIRS["final"] / f".{mkv_final_path.stem}.{run_id}.staged.mkv"
    staged_paths = [
        tr_stage, id_stage, tr_sidecar_stage, id_sidecar_stage, review_stage, mkv_stage
    ]
    mux_report = None
    prior_report_backup = None
    superseded_mkv_path = None
    try:
        write_srt(tr_stage, tr_entries)
        write_srt(id_stage, id_entries)
        write_srt(tr_sidecar_stage, tr_entries)
        write_srt(id_sidecar_stage, id_entries)
        assert_srt_roundtrip(tr_stage, tr_entries)
        assert_srt_roundtrip(id_stage, id_entries)
        assert_srt_roundtrip(tr_sidecar_stage, tr_entries)
        assert_srt_roundtrip(id_sidecar_stage, id_entries)
        tr_readback = parse_srt(tr_stage)
        id_readback = parse_srt(id_stage)
        assert_entry_contract(tr_readback, schema["blocks"], "TR readback", hard_limit)
        assert_entry_contract(id_readback, schema["blocks"], "ID readback", hard_limit)
        assert_identical_timings(tr_readback, id_readback)

        create_review_xlsx(
            review_stage,
            schema["blocks"],
            validation,
            qa_report,
            max_fraction=0.05,
        )
        review_row_count = verify_review_xlsx(review_stage)
        explicit_review_rows = validation_report["review_required_count"]
        maximum_review_rows = max(1, (schema["block_count"] + 19) // 20)
        if review_row_count < explicit_review_rows:
            raise RuntimeError(
                f"Review workbook dropped explicit review requests: "
                f"{review_row_count} < {explicit_review_rows}"
            )
        allowed_review_rows = max(explicit_review_rows, maximum_review_rows)
        if review_row_count > allowed_review_rows:
            raise RuntimeError(
                f"Review workbook exceeds the 5% target/explicit-request floor: "
                f"{review_row_count} > {allowed_review_rows}"
            )

        if CREATE_MKV:
            mux_report = mux_softsubs(
                source_video,
                id_stage,
                tr_stage,
                mkv_stage,
                verify_stream_hashes=True,
            )
            if not (
                mux_report.get("verified") is True
                and mux_report.get("video_audio_stream_copy") is True
                and mux_report.get("subtitle_order") == ["ind", "tur"]
                and mux_report.get("indonesian_default") is True
                and mux_report.get("turkish_default") is False
                and mux_report.get("roundtrip", {}).get("exact") is True
                and mux_report.get("roundtrip", {}).get("id_block_count") == schema["block_count"]
                and mux_report.get("roundtrip", {}).get("tr_block_count") == schema["block_count"]
                and mux_report.get("stream_hashes", {}).get("checked") is True
                and mux_report.get("stream_hashes", {}).get("match") is True
            ):
                raise RuntimeError("MKV verification report is incomplete or failed")

        # Recheck every captured input after potentially long mux/hash work,
        # immediately before changing any public artifact or commit marker.
        assert_current_input_hashes("before publishing final outputs")
        # Invalidate an older commit marker before replacing its artifacts.
        if finalization_report_path.is_file():
            prior_report_backup = (
                DIRS["final"]
                / f".{finalization_report_path.stem}.{run_id}.superseded.json"
            )
            os.replace(finalization_report_path, prior_report_backup)

        if not CREATE_MKV and mkv_final_path.exists():
            require_file(mkv_final_path, "Stale canonical MKV")
            superseded_mkv_path = (
                DIRS["final"]
                / f".{mkv_final_path.stem}.{run_id}.superseded.mkv"
            )
            os.replace(mkv_final_path, superseded_mkv_path)

        # Publish only after every staged artifact and optional MKV has passed.
        atomic_publish(tr_stage, tr_final_path)
        atomic_publish(id_stage, id_final_path)
        atomic_publish(tr_sidecar_stage, tr_sidecar_path)
        atomic_publish(id_sidecar_stage, id_sidecar_path)
        atomic_publish(review_stage, review_final_path)
        if CREATE_MKV:
            atomic_publish(mkv_stage, mkv_final_path)

        output_records = {key: file_record(path, EPISODE_ROOT) for key, path in output_paths.items()}
        input_file_records = report_input_file_records()
        if mux_report is not None:
            mux_report = dict(mux_report)
            mux_report["output_path"] = output_records["mkv"]["relative_path"]
        finalization_report = {
            "report_version": 1,
            "status": "PASS",
            "completed_at": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
            "episode": EPISODE,
            "episode_name": episode_name,
            "schema_version": schema["schema_version"],
            "schema_sha256": schema["schema_sha256"],
            "block_count": schema["block_count"],
            "input_hashes": input_hashes,
            "input_files": input_file_records,
            "settings": {"create_mkv": CREATE_MKV, "strict_qa": STRICT_QA},
            "translation_validation": validation_report,
            "subtitle_qa": qa_report,
            "timing_identity": {
                "schema_only": True,
                "tr_id_identical": True,
                "srt_roundtrip_exact": True,
            },
            "review_row_count": review_row_count,
            "mkv_verification": mux_report,
            "outputs": output_records,
        }
        # The report is the last atomic write and therefore the commit marker.
        assert_current_input_hashes("before writing the PASS commit marker")
        atomic_write_json(finalization_report_path, finalization_report)
        report_readback = read_json(finalization_report_path)
        if report_readback != finalization_report:
            raise RuntimeError("Finalization report JSON readback mismatch")
        if prior_report_backup is not None:
            prior_report_backup.unlink(missing_ok=True)
        if superseded_mkv_path is not None:
            superseded_mkv_path.unlink(missing_ok=True)
    finally:
        for staged_path in staged_paths:
            staged_path.unlink(missing_ok=True)

print("")
print("FINALIZATION PASS")
print(f"TR SRT: {tr_final_path}")
print(f"ID SRT: {id_final_path}")
print(f"Infuse TR sidecar: {tr_sidecar_path}")
print(f"Infuse ID sidecar: {id_sidecar_path}")
print(f"Review: {review_final_path}")
if CREATE_MKV:
    print(f"MKV: {mkv_final_path}")
print(f"Report: {finalization_report_path}")
print(f"Blocks: {schema['block_count']:,}; positional mismatches: 0; overlaps: 0")